# ETL MIO | Extractos Silver, Gold y KPIs

Conversión de `Avance2.py` a notebook. Ejecutar en orden: configuración → tablas documentales → consultas → controles → gráficos. Los resultados de PostgreSQL aparecen como tablas dentro del notebook. También se exportan CSV y PNG.

**Dependencias:** `pandas`, `matplotlib`, `SQLAlchemy`, `psycopg[binary]`, `ipykernel`. Configure `DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER` y `DB_PASSWORD` en el entorno antes de abrir Jupyter. No escriba contraseñas dentro del notebook. La base debe contener las tablas y vistas que se consultan abajo.

El notebook guardado se entrega sin resultados simulados: al ejecutar las celdas, los resultados reales quedan visibles y pueden guardarse con el archivo `.ipynb`.


In [ ]:
from pathlib import Path
from typing import Optional
import os
import pandas as pd
import matplotlib as plt
#from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

%matplotlib inline

# Si el notebook está en reportes/, este directorio coincide con Avance2.py.
BASE_DIR = Path.cwd() if Path.cwd().name == 'reportes' else Path.cwd() / 'reportes'
OUTPUT_DIR = BASE_DIR / 'salidas'
SILVER_DIR = OUTPUT_DIR / 'silver'
GOLD_DIR = OUTPUT_DIR / 'gold'
KPI_DIR = OUTPUT_DIR / 'kpi'
GRAFICOS_DIR = OUTPUT_DIR / 'graficos'
for directory in (SILVER_DIR, GOLD_DIR, KPI_DIR, GRAFICOS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# None muestra todas las filas en Jupyter; ajuste a un número si las tablas son grandes.
MAX_FILAS_VISTA = 200

def mostrar_tabla(nombre: str, df: pd.DataFrame) -> None:
    display(Markdown(f'### {nombre} · {len(df):,} filas × {len(df.columns)} columnas'))
    if df.empty:
        display(Markdown('Sin filas disponibles para esta consulta.'))
        return
    with pd.option_context('display.max_rows', MAX_FILAS_VISTA, 'display.max_columns', None):
        display(df)
    if MAX_FILAS_VISTA is not None and len(df) > MAX_FILAS_VISTA:
        display(Markdown(f'Vista limitada a {MAX_FILAS_VISTA} filas; el CSV conserva todas. Cambie `MAX_FILAS_VISTA = None` para visualizar todas.'))


In [ ]:
import sys
print(sys.executable)

## Fuentes documentadas y metas propuestas

Estas tablas proceden del código; no se presentan como resultados medidos en la base.

In [ ]:
PERFIL_COORDENADAS = pd.DataFrame(
    [
        {"hoja": "PSO 260316", "fecha_inicio": "2026-03-16", "filas_validas": 2072},
    ]
)


KR_PROPUESTOS = pd.DataFrame(
    [
        {
            "KR": "KR1",
            "Resultado_clave": (
                "Fuentes del alcance ejecutable con archivo/version/"
                "fecha/mecanismo documentado"
            ),
            "Meta_propuesta": "100%",
            "Evidencia": "Inventario de fuentes + carga auditada",
        },
        {
            "KR": "KR2",
            "Resultado_clave": (
                "Registros de coordenadas con código, latitud y longitud válidos"
            ),
            "Meta_propuesta": ">= 99%",
            "Evidencia": "gold.kpi_coordenadas",
        },
        {
            "KR": "KR3",
            "Resultado_clave": "Datasets cargados sin pérdida no explicada de filas",
            "Meta_propuesta": "100% de cargas exitosas",
            "Evidencia": "filas_leidas = filas_cargadas + filas_rechazadas",
        },
        {
            "KR": "KR4",
            "Resultado_clave": "Reejecución del mismo archivo detectada como duplicado",
            "Meta_propuesta": "100%",
            "Evidencia": "sha256_archivo / sha256_contenido",
        },
        {
            "KR": "KR5",
            "Resultado_clave": "Paradas GTFS asignadas espacialmente a comuna",
            "Meta_propuesta": ">= 95%",
            "Evidencia": "No evaluable hasta cargar GTFS + límites",
        },
        {
            "KR": "KR6",
            "Resultado_clave": "Llaves ruta-parada válidas después de normalización",
            "Meta_propuesta": ">= 99%",
            "Evidencia": "Silver + controles de integridad",
        },
    ]
)


INVENTARIO_FUENTES = pd.DataFrame(
    [
        {
            "fuente": "USOS / USOS_VALIDADOR",
            "estado_actual": "Implementada",
            "version_corte": "Por archivo de recaudo",
            "mecanismo": "Carga Excel -> Python -> PostgreSQL",
        },
        {
            "fuente": "DIA_TIPO",
            "estado_actual": "Implementada",
            "version_corte": "Según archivo de calendario",
            "mecanismo": "Carga Excel -> Python -> PostgreSQL",
        },
        {
            "fuente": "PSO",
            "estado_actual": "Implementada",
            "version_corte": "Vigencia obtenida del nombre PSO_YYMMDD",
            "mecanismo": "Carga Excel multihoja -> Python -> PostgreSQL",
        },
        {
            "fuente": "COORDENADAS_PARADAS",
            "estado_actual": "Incorporada en este avance",
            "version_corte": "Snapshots PSO 251124 a PSO 260803",
            "mecanismo": "Excel multihoja -> Python -> PostgreSQL",
        },
        
        #Pendiente para carga futura de fuentes espaciales y poblacionales:
        #{
        #    "fuente": "GTFS estático",
        #    "estado_actual": "Pendiente",
        #    "version_corte": "Por documentar",
        #    "mecanismo": "Debe definirse mecanismo reproducible",
        #},
        #
        #{
        #    "fuente": "Límites de comunas",
        #    "estado_actual": "Pendiente",
        #    "version_corte": "Por documentar",
        #    "mecanismo": "Debe definirse archivo/servicio y fecha de corte",
        #},
        #{
        #    "fuente": "Población",
        #    "estado_actual": "Pendiente",
        #    "version_corte": "Por documentar",
        #    "mecanismo": "Debe definirse fuente, año y versión",
        #},
        
    ]
)


PERFIL_COORDENADAS['fecha_inicio'] = pd.to_datetime(PERFIL_COORDENADAS['fecha_inicio']).dt.date

for nombre, tabla in (("Perfil documentado de coordenadas", PERFIL_COORDENADAS), ("Resultados clave propuestos", KR_PROPUESTOS), ("Inventario de fuentes", INVENTARIO_FUENTES)):
    mostrar_tabla(nombre, tabla)


In [ ]:
import sys
print(sys.executable)

## Conexión y utilidades

SQLAlchemy administra la conexión utilizada por `pandas`. Cada consulta se ejecuta con una conexión nueva del mismo engine, por lo que un fallo puntual no invalida las consultas siguientes.

In [ ]:
REQUIRED_ENV = ("DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD")

def validate_environment() -> None:
    missing = [key for key in REQUIRED_ENV if not os.getenv(key)]

    if missing:
        raise RuntimeError(
            "Faltan variables de entorno para la conexión: "
            + ", ".join(missing)
        )


def get_engine():
    validate_environment()

    url = URL.create(
        drivername="postgresql+psycopg",
        username=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT", "5432")),
        database=os.getenv("DB_NAME"),
        query={"sslmode": "require"},
    )

    return create_engine(
        url,
        pool_pre_ping=True,
    )


def query_or_empty(conn, sql: str, label: Optional[str] = None) -> pd.DataFrame:
    try:
        return pd.read_sql_query(sql, conn)
    except Exception as exc:
        # Una consulta opcional fallida no debe ocultar las demás vistas.
        print(f"Consulta no disponible [{label or 'sin nombre'}]: {exc}")
        return pd.DataFrame()

def export_if_data(df: pd.DataFrame, path: Path) -> None:
    if df.empty:
        print(f'Sin datos para exportar: {path.name}')
        return
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'Exportado: {path}')

def format_thousands(value, _position) -> str:
    try:
        return f'{int(value):,}'.replace(',', '.')
    except (TypeError, ValueError):
        return str(value)


## Consultas de auditoría, Silver y Gold

Definiciones SQL originales del archivo adjunto.

In [ ]:
SQL_AUDITORIA = """
select
    a.id as archivo_id,
    a.nombre_archivo,
    a.tipo_archivo,
    a.estado as estado_archivo,
    a.fecha_archivo,
    a.iniciado_en,
    a.finalizado_en,
    d.id as dataset_id,
    d.tipo_dataset,
    d.estado as estado_dataset,
    d.filas_leidas,
    d.filas_cargadas,
    d.filas_rechazadas,
    d.fecha_min,
    d.fecha_max
from carga.archivo_etl a
left join carga.dataset_etl d
    on d.archivo_id = a.id
order by
    a.iniciado_en,
    d.tipo_dataset
"""


SQL_SILVER_USOS_HORA = """
select *
from silver.fact_usos_hora
order by fecha, estacion, hora
"""


SQL_SILVER_PARADAS = """
select *
from silver.dim_parada_actual
order by codigo_parada
"""


SQL_SILVER_RUTA_PARADA = """
select *
from silver.puente_ruta_parada
order by codigo_ruta, codigo_parada
"""


SQL_GOLD_DEMANDA_DIARIA = """
select *
from gold.demanda_diaria
order by fecha
"""


SQL_GOLD_DEMANDA_ESTACION = """
select *
from gold.demanda_estacion_dia_tipo
order by estacion, dia_tipo
"""


SQL_KPI_DATASET = """
select *
from gold.kpi_carga_dataset
order by tipo_dataset
"""


SQL_KPI_ARCHIVO = """
select *
from gold.kpi_carga_archivo
order by tipo_archivo
"""


SQL_KPI_COORDENADAS = """
select *
from gold.kpi_coordenadas
order by fecha_inicio_vigencia
"""


## Lectura de resultados

Ejecute esta celda para consultar y mostrar cada tabla. Los DataFrames quedan disponibles con los nombres indicados en el código.

In [ ]:
resultados = {}
consultas = [
    ('auditoria', 'Auditoría de cargas', SQL_AUDITORIA, KPI_DIR / 'auditoria_cargas.csv'),
    ('silver_usos_hora', 'Silver: usos por hora', SQL_SILVER_USOS_HORA, SILVER_DIR / 'fact_usos_hora.csv'),
    ('silver_paradas', 'Silver: paradas actuales', SQL_SILVER_PARADAS, SILVER_DIR / 'dim_parada_actual.csv'),
    ('silver_ruta_parada', 'Silver: ruta y parada', SQL_SILVER_RUTA_PARADA, SILVER_DIR / 'puente_ruta_parada.csv'),
    ('gold_demanda_diaria', 'Gold: demanda diaria', SQL_GOLD_DEMANDA_DIARIA, GOLD_DIR / 'demanda_diaria.csv'),
    ('gold_demanda_estacion', 'Gold: demanda por estación y día tipo', SQL_GOLD_DEMANDA_ESTACION, GOLD_DIR / 'demanda_estacion_dia_tipo.csv'),
    ('kpi_dataset', 'KPI: carga por dataset', SQL_KPI_DATASET, KPI_DIR / 'kpi_carga_dataset.csv'),
    ('kpi_archivo', 'KPI: carga por archivo', SQL_KPI_ARCHIVO, KPI_DIR / 'kpi_carga_archivo.csv'),
    ('kpi_coordenadas', 'KPI: coordenadas', SQL_KPI_COORDENADAS, KPI_DIR / 'kpi_coordenadas.csv'),
]

engine = get_engine()
try:
    for variable, titulo, sql, archivo in consultas:
        with engine.connect() as conn:
            df = query_or_empty(conn, sql, titulo)
        resultados[variable] = df
        globals()[variable] = df
        mostrar_tabla(titulo, df)
        export_if_data(df, archivo)
finally:
    engine.dispose()

for nombre, tabla, archivo in (
    ('Perfil documentado de coordenadas', PERFIL_COORDENADAS, KPI_DIR / 'perfil_coordenadas_documentado.csv'),
    ('KR propuestos', KR_PROPUESTOS, KPI_DIR / 'kr_propuestos.csv'),
    ('Inventario de fuentes', INVENTARIO_FUENTES, KPI_DIR / 'inventario_fuentes.csv'),
):
    export_if_data(tabla, archivo)


## Controles de consistencia e idempotencia

Calculados a partir del resultado de auditoría; se muestran como tablas y se exportan completos a CSV.

In [ ]:
def construir_consistencia(
    auditoria: pd.DataFrame,
) -> pd.DataFrame:
    required = {
        "dataset_id",
        "tipo_dataset",
        "filas_leidas",
        "filas_cargadas",
        "filas_rechazadas",
    }

    if auditoria.empty or not required.issubset(auditoria.columns):
        return pd.DataFrame()

    df = auditoria[
        auditoria["dataset_id"].notna()
    ].copy()

    if df.empty:
        return pd.DataFrame()

    for column in (
        "filas_leidas",
        "filas_cargadas",
        "filas_rechazadas",
    ):
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        ).fillna(0)

    df["diferencia"] = (
        df["filas_leidas"]
        - df["filas_cargadas"]
        - df["filas_rechazadas"]
    )

    df["consistente"] = (
        df["diferencia"] == 0
    )

    return (
        df.groupby(
            [
                "tipo_dataset",
                "consistente",
            ],
            dropna=False,
        )
        .size()
        .reset_index(name="datasets")
    )


def construir_duplicados(
    auditoria: pd.DataFrame,
) -> pd.DataFrame:
    required = {
        "tipo_archivo",
        "tipo_dataset",
        "estado_archivo",
        "estado_dataset",
    }

    if auditoria.empty or not required.issubset(auditoria.columns):
        return pd.DataFrame()

    df = auditoria[
        (
            auditoria["estado_archivo"]
            == "DUPLICADO"
        )
        |
        (
            auditoria["estado_dataset"]
            == "DUPLICADO"
        )
    ].copy()

    if df.empty:
        return pd.DataFrame()

    return (
        df.groupby(
            [
                "tipo_archivo",
                "tipo_dataset",
            ],
            dropna=False,
        )
        .size()
        .reset_index(
            name="duplicados_detectados"
        )
    )


In [ ]:
consistencia = construir_consistencia(auditoria)
duplicados = construir_duplicados(auditoria)
for nombre, tabla, archivo in (
    ('Consistencia de filas', consistencia, KPI_DIR / 'consistencia_filas.csv'),
    ('Duplicados detectados', duplicados, KPI_DIR / 'duplicados_detectados.csv'),
):
    mostrar_tabla(nombre, tabla)
    export_if_data(tabla, archivo)


## Gráficos integrados

Las figuras se ven aquí y se guardan como PNG. Las etiquetas muestran valores en barras, porcentajes en coordenadas, mínimo/máximo/último de la demanda y mediana en el boxplot.

In [ ]:
def grafico_filas_dataset(
    df: pd.DataFrame,
) -> None:
    required = {
        "tipo_dataset",
        "filas_leidas",
        "filas_cargadas",
        "filas_rechazadas",
    }

    if df.empty or not required.issubset(df.columns):
        return

    plot_df = df[
        [
            "tipo_dataset",
            "filas_leidas",
            "filas_cargadas",
            "filas_rechazadas",
        ]
    ].set_index("tipo_dataset")

    ax = plot_df.plot(
        kind="barh",
        figsize=(10, 5),
    )

    for container in ax.containers:
        ax.bar_label(container, labels=[format_thousands(v, None) if v > 0 else '' for v in container.datavalues], padding=3, fontsize=8)
    ax.set_xlim(right=ax.get_xlim()[1] * 1.18)
    ax.set_title("Volumen procesado por dataset")
    ax.set_xlabel("Filas")
    ax.set_ylabel("Dataset")
    ax.xaxis.set_major_formatter(
        FuncFormatter(format_thousands)
    )

    plt.tight_layout()
    plt.savefig(
        GRAFICOS_DIR / "01_volumen_procesado_dataset.png",
        dpi=220,
        bbox_inches="tight",
    )
    display(plt.gcf())
    plt.close()


def grafico_estado_archivos(
    df: pd.DataFrame,
) -> None:
    required = {
        "tipo_archivo",
        "archivos_cargados",
        "archivos_parciales",
        "archivos_duplicados",
        "archivos_error",
    }

    if df.empty or not required.issubset(df.columns):
        return

    plot_df = df[
        [
            "tipo_archivo",
            "archivos_cargados",
            "archivos_parciales",
            "archivos_duplicados",
            "archivos_error",
        ]
    ].set_index("tipo_archivo")

    ax = plot_df.plot(
        kind="bar",
        stacked=True,
        figsize=(10, 5),
    )

    for container in ax.containers:
        ax.bar_label(container, labels=[format_thousands(v, None) if v > 0 else '' for v in container.datavalues], label_type='center', fontsize=8)
    ax.set_title("Resultado de las ejecuciones ETL por fuente")
    ax.set_xlabel("Tipo de archivo")
    ax.set_ylabel("Archivos")
    ax.yaxis.set_major_formatter(
        FuncFormatter(format_thousands)
    )

    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    plt.savefig(
        GRAFICOS_DIR / "02_estado_ejecuciones_fuente.png",
        dpi=220,
        bbox_inches="tight",
    )
    display(plt.gcf())
    plt.close()


def grafico_coordenadas(
    df: pd.DataFrame,
) -> None:
    required = {
        "fecha_inicio_vigencia",
        "pct_coordenadas_validas",
    }

    if df.empty or not required.issubset(df.columns):
        return

    plot_df = df.copy()
    plot_df["fecha_inicio_vigencia"] = pd.to_datetime(
        plot_df["fecha_inicio_vigencia"]
    )

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(
        plot_df["fecha_inicio_vigencia"],
        plot_df["pct_coordenadas_validas"],
        marker="o",
    )

    for x, y in zip(plot_df['fecha_inicio_vigencia'], plot_df['pct_coordenadas_validas']):
        ax.annotate(f'{y:.1f}%', (x, y), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=8)
    ax.axhline(99, linestyle="--")
    ax.set_ylim(0, 100)
    ax.set_title("Calidad de georreferenciación por vigencia")
    ax.set_xlabel("Fecha de inicio de vigencia")
    ax.set_ylabel("Coordenadas válidas (%)")

    plt.tight_layout()

    plt.savefig(
        GRAFICOS_DIR / "03_coordenadas_validas_snapshot.png",
        dpi=220,
        bbox_inches="tight",
    )
    display(plt.gcf())
    plt.close()


def grafico_demanda_diaria(
    df: pd.DataFrame,
) -> None:
    required = {
        "fecha",
        "demanda_total",
    }

    if df.empty or not required.issubset(df.columns):
        return

    plot_df = df.copy()
    plot_df["fecha"] = pd.to_datetime(
        plot_df["fecha"]
    )

    fig, ax = plt.subplots(figsize=(11, 5))

    ax.plot(
        plot_df["fecha"],
        plot_df["demanda_total"],
        marker="o",
        markersize=3,
    )

    valid = plot_df.dropna(subset=['fecha', 'demanda_total'])
    if not valid.empty:
        for idx in {valid['demanda_total'].idxmin(), valid['demanda_total'].idxmax(), valid.index[-1]}:
            row = valid.loc[idx]
            ax.annotate(format_thousands(row['demanda_total'], None), (row['fecha'], row['demanda_total']), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=8)
    ax.set_title("Demanda diaria del periodo cargado")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Usos")
    ax.yaxis.set_major_formatter(
        FuncFormatter(format_thousands)
    )

    plt.tight_layout()

    plt.savefig(
        GRAFICOS_DIR / "04_demanda_diaria.png",
        dpi=220,
        bbox_inches="tight",
    )
    display(plt.gcf())
    plt.close()


def grafico_demanda_dia_tipo(
    df: pd.DataFrame,
) -> None:
    required = {
        "dia_tipo",
        "demanda_total",
    }

    if df.empty or not required.issubset(df.columns):
        return

    plot_df = df[
        df["dia_tipo"].notna()
    ].copy()

    if plot_df.empty:
        return

    categorias = list(
        plot_df["dia_tipo"]
        .astype(str)
        .drop_duplicates()
    )

    data = [
        plot_df.loc[
            plot_df["dia_tipo"].astype(str) == categoria,
            "demanda_total",
        ]
        .dropna()
        .to_numpy()
        for categoria in categorias
    ]

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.boxplot(
        data,
        tick_labels=categorias,
    )

    for idx, values in enumerate(data, start=1):
        if len(values):
            ax.annotate(format_thousands(pd.Series(values).median(), None), (idx, pd.Series(values).median()), xytext=(6, 2), textcoords='offset points', fontsize=8)
    ax.set_title("Distribución de demanda según día tipo")
    ax.set_xlabel("Día tipo")
    ax.set_ylabel("Demanda diaria")
    ax.yaxis.set_major_formatter(
        FuncFormatter(format_thousands)
    )

    plt.tight_layout()

    plt.savefig(
        GRAFICOS_DIR / "05_demanda_por_dia_tipo.png",
        dpi=220,
        bbox_inches="tight",
    )
    display(plt.gcf())
    plt.close()


In [ ]:
grafico_filas_dataset(kpi_dataset)
grafico_estado_archivos(kpi_archivo)
grafico_coordenadas(kpi_coordenadas)
grafico_demanda_diaria(gold_demanda_diaria)
grafico_demanda_dia_tipo(gold_demanda_diaria)
print(f'CSV y PNG: {OUTPUT_DIR}')
